This Notebook specifically was run on codespaces and might not run in local.
This was done because lateralJoin was not available as a DataFrame function in my local spark setup (Version 3.5.0).
Because of this the setup of tables is done seperately for this notebook again.

In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/local/sdkman/candidates/java/17.0.10-ms"
os.environ["JAVA_TOOL_OPTIONS"] = " ".join([
    "--add-opens=java.base/javax.security.auth=ALL-UNNAMED",
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED",
    "--add-opens=java.base/java.nio=ALL-UNNAMED",
    "--add-opens=java.base/java.lang=ALL-UNNAMED",
    "--add-opens=java.base/java.util=ALL-UNNAMED",
])

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/workspaces/pyspark_udemy_codespace/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/workspaces/pyspark_udemy_codespace/metastore_db;create=true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/08 13:43:30 WARN Utils: Your hostname, codespaces-9ec455, resolves to a loopback address: 127.0.0.1; using 10.0.1.161 instead (on interface eth0)
26/06/08 13:43:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(

Spark version: 4.1.1


Lateral Join
    Lateral join allows to query right dataframe for each row of the left dataframe.
    Lateral joins are especially useful when:
        You need per-parent Top-N child rows
        You want to invoke TVFs with arguments derived from each row

In [2]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [3]:
spark.sql("create database if not exists spark_db")

DataFrame[]

In [4]:
"""
Load data from files and create the following tables.
    facilities -> facilities.csv
    members -> members.csv
    bookings -> bookings.csv
Choose appropriate data types to best represent the data fields.
"""
# defining schemas

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType # type: ignore

facilities_schema = StructType([
    StructField("facid", IntegerType(), nullable=False), # though we have mentioned nullable False here the schema does not show that because in spark constraints during read are not strict constraints, they are parsing instructions
    StructField("fac_name", StringType()),
    StructField("membercost", IntegerType()),
    StructField("guestcost", IntegerType()),
    StructField("initialoutlay", IntegerType()),
    StructField("monthlymaintenance", IntegerType())
])

members_schema = StructType([
    StructField("memid", IntegerType(), nullable=False), # though we have mentioned nullable False here the schema does not show that because in spark constraints during read are not strict constraints, they are parsing instructions
    StructField("surname", StringType()),
    StructField("firstname", StringType()),
    StructField("address", StringType()),
    StructField("zipcode", StringType()),
    StructField("telephone", StringType()),
    StructField("recommendedby", IntegerType()),
    StructField("joindate", TimestampType())
])

bookings_schema = StructType([
    StructField("bookid", IntegerType(), nullable=False), # though we have mentioned nullable False here the schema does not show that because in spark constraints during read are not strict constraints, they are parsing instructions
    StructField("facid", IntegerType()),
    StructField("memid", IntegerType()),
    StructField("starttime", TimestampType()),
    StructField("slots", IntegerType())
])

In [5]:
# loading into dataframe

facilities_df = spark.read.format("csv")\
                        .option("header", True)\
                        .schema(facilities_schema)\
                        .load(path = "/workspaces/pyspark_udemy_codespace/data/facilities.csv")
facilities_df.show()
facilities_df.printSchema()

+-----+---------------+----------+---------+-------------+------------------+
|facid|       fac_name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+---------------+----------+---------+-------------+------------------+
|    0| Tennis Court 1|         5|       25|        10000|               200|
|    1| Tennis Court 2|         5|       25|         8000|               200|
|    2|Badminton Court|         0|     NULL|         4000|                50|
|    3|   Table Tennis|         0|        5|          320|                10|
|    4| Massage Room 1|        35|       80|         4000|              3000|
|    5| Massage Room 2|        35|       80|         4000|              3000|
|    6|   Squash Court|      NULL|     NULL|         5000|                80|
|    7|  Snooker Table|         0|        5|          450|                15|
|    8|     Pool Table|         0|        5|          400|                15|
+-----+---------------+----------+---------+-------------+------

In [6]:
members_df = spark.read.format("csv")\
                    .option("header", True)\
                    .schema(members_schema)\
                    .load(path = "/workspaces/pyspark_udemy_codespace/data/members.csv")
members_df.show()
members_df.printSchema()

+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|memid|  surname|firstname|             address|zipcode|     telephone|recommendedby|           joindate|
+-----+---------+---------+--------------------+-------+--------------+-------------+-------------------+
|    0|    GUEST|    GUEST|               GUEST|      0|(000) 000-0000|         NULL|2022-07-01 00:00:00|
|    1|    Smith|   Darren|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2022-07-02 12:02:05|
|    2|    Smith|    Tracy|8 Bloomsbury Clos...|   4321|  555-555-5555|         NULL|2022-07-02 12:08:23|
|    3|   Rownam|      Tim|23 Highway Way, B...|  23423|(844) 693-0723|         NULL|2022-07-03 09:32:15|
|    4| Joplette|   Janice|20 Crossing Road,...|    234|(833) 942-4710|            1|2022-07-03 10:25:05|
|    5|  Butters|   Gerald|1065 Huntingdon A...|  56754|(844) 078-4130|            1|2022-07-09 10:44:09|
|    6|    Tracy|   Burton|3 Tunisia Drive, ..

In [7]:
bookings_df = spark.read.format("csv")\
                    .option("header", True)\
                    .schema(bookings_schema)\
                    .load(path = "/workspaces/pyspark_udemy_codespace/data/bookings.csv")
bookings_df.show()
bookings_df.printSchema()

+------+-----+-----+-------------------+-----+
|bookid|facid|memid|          starttime|slots|
+------+-----+-----+-------------------+-----+
|     0|    3|    1|2022-07-03 11:00:00|    2|
|     1|    4|    1|2022-07-03 08:00:00|    2|
|     2|    6|    0|2022-07-03 18:00:00|    2|
|     3|    7|    1|2022-07-03 19:00:00|    2|
|     4|    8|    1|2022-07-03 10:00:00|    1|
|     5|    8|    1|2022-07-03 15:00:00|    1|
|     6|    0|    2|2022-07-04 09:00:00|    3|
|     7|    0|    2|2022-07-04 15:00:00|    3|
|     8|    4|    3|2022-07-04 13:30:00|    2|
|     9|    4|    0|2022-07-04 15:00:00|    2|
|    10|    4|    0|2022-07-04 17:30:00|    2|
|    11|    6|    0|2022-07-04 12:30:00|    2|
|    12|    6|    0|2022-07-04 14:00:00|    2|
|    13|    6|    1|2022-07-04 15:30:00|    2|
|    14|    7|    2|2022-07-04 14:00:00|    2|
|    15|    8|    2|2022-07-04 12:00:00|    1|
|    16|    8|    3|2022-07-04 18:00:00|    1|
|    17|    1|    0|2022-07-05 17:30:00|    3|
|    18|    2

In [9]:
# loading dfs into table

facilities_df.write.mode("overwrite").saveAsTable("spark_db.facilities")
members_df.write.mode("overwrite").saveAsTable("spark_db.members")
bookings_df.write.mode("overwrite").saveAsTable("spark_db.bookings")

In [10]:
spark.sql("SELECT * FROM spark_db.facilities").show()
spark.sql("SELECT * FROM spark_db.members").show()
spark.sql("SELECT * FROM spark_db.bookings").show()

+-----+---------------+----------+---------+-------------+------------------+
|facid|       fac_name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+---------------+----------+---------+-------------+------------------+
|    0| Tennis Court 1|         5|       25|        10000|               200|
|    1| Tennis Court 2|         5|       25|         8000|               200|
|    2|Badminton Court|         0|     NULL|         4000|                50|
|    3|   Table Tennis|         0|        5|          320|                10|
|    4| Massage Room 1|        35|       80|         4000|              3000|
|    5| Massage Room 2|        35|       80|         4000|              3000|
|    6|   Squash Court|      NULL|     NULL|         5000|                80|
|    7|  Snooker Table|         0|        5|          450|                15|
|    8|     Pool Table|         0|        5|          400|                15|
+-----+---------------+----------+---------+-------------+------

In [16]:
'''
Find the most recent booking for each member
+---------+----------+---------+---------------+-------------------+-----+
|member_id|first_name|last_name|  facility_name|         start_time|slots|
+---------+----------+---------+---------------+-------------------+-----+
'''
from pyspark.sql.functions import col

members_df = spark.table("spark_db.members")\
                .filter(col("memid") > 0)\
                .select(
                    col("memid"),
                    col("firstname"),
                    col("surname")
                )\
                .alias("m")
bookings_df = spark.table("spark_db.bookings").alias("b")
facilities_df = spark.table("spark_db.facilities").alias("f")

latest_member_booking_df = members_df.lateralJoin(
        bookings_df.where("b.memid == m.memid").orderBy(col("start_time").desc()).limit(1), # this is the join condition here and the order by and limit
        None, # No join expression since the condition is already mentioned in the where clause while filtering from the left DF
        "left"
    )\
    .select(col("m.memid"), col("m.firstname").alias("first_name"), col("m.surname").alias("last_name"), col("b.facid").alias("facility_id"), col("b.start_time"), col("b.slots"))
latest_member_booking_df.show()

# essentially lateral join joins to DFs like any other join and the orders by some param and the picks based on the limit and returns the resultant DF
# this join is best to solve "top N" problems

{"ts": "2026-06-08 14:05:45.369", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `m`.`memid` cannot be resolved. Did you mean one of the following? [`b`.`memid`, `b`.`facid`, `b`.`bookid`, `b`.`slots`, `b`.`starttime`]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o244.filter.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `m`.`memid` cannot be resolved. Did you mean one of the following? [`b`.`memid`, `b`.`facid`, `b`.`bookid`, `b`.`slots`, `b`.`starttime`]. SQLSTATE: 42703; line 1 pos 11;\n'Filter (memid#187 = 'm.memid)\n+- SubqueryAlias b\n   +- SubqueryAlias spark_catalog.spark_db.bookings\n      +- Relation spark_catalog.spark_db.bookings[bookid#185,facid#186,memid#187,starttime#188,

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `m`.`memid` cannot be resolved. Did you mean one of the following? [`b`.`memid`, `b`.`facid`, `b`.`bookid`, `b`.`slots`, `b`.`starttime`]. SQLSTATE: 42703; line 1 pos 11;
'Filter (memid#187 = 'm.memid)
+- SubqueryAlias b
   +- SubqueryAlias spark_catalog.spark_db.bookings
      +- Relation spark_catalog.spark_db.bookings[bookid#185,facid#186,memid#187,starttime#188,slots#189] parquet
